# SemiRestoreNet — KLA SEMICON Hackathon 2026
## Joint Semiconductor Image Denoising + 2x Super-Resolution

**Instructions:**
1. **Runtime > Change runtime type > T4 GPU** (free tier)
2. Upload your `train.zip` and `Test_NoisyLR.zip` to Google Drive under `My Drive/SemiconHackathon/Dataset/`
3. Run all cells in order (`Runtime > Run all`)
4. Best model auto-saves to Drive — download `best_model.pth`

**Expected training time:** ~30-60 min for 100 epochs on T4 GPU

## Step 1: Check GPU & Mount Google Drive

In [ ]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found! Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted at /content/drive')

## Step 2: Install Dependencies

In [ ]:
!pip install -q torch torchvision numpy matplotlib scikit-image pillow tqdm onnx onnxruntime onnxscript
print('Dependencies installed')

## Step 3: Set Up Paths & Extract Dataset

Before running this cell, upload your files to Google Drive:
```
My Drive/
  SemiconHackathon/
    Dataset/
      train.zip            (training data ~919 MB)
      Test_NoisyLR.zip     (test data ~23 MB)
```

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive/SemiconHackathon')
DATASET_DIR  = DRIVE_ROOT / 'Dataset'
WORK_DIR     = Path('/content/work_stage1')
OUTPUT_DIR   = DRIVE_ROOT / 'checkpoints'

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_NOISY_DIR = WORK_DIR / 'train' / 'NoisyLR'
TRAIN_GT_DIR    = WORK_DIR / 'train' / 'GT'
TEST_NOISY_DIR  = WORK_DIR / 'NoisyLR'

train_zip = DATASET_DIR / 'train.zip'
test_zip  = DATASET_DIR / 'Test_NoisyLR.zip'

print(f'Drive root  : {DRIVE_ROOT}')
print(f'Output dir  : {OUTPUT_DIR}')
print(f'train.zip   : {"FOUND" if train_zip.exists() else "NOT FOUND"}')
print(f'Test zip    : {"FOUND" if test_zip.exists() else "NOT FOUND (optional)"}')

In [ ]:
import zipfile, time

def extract_if_needed(zip_path, dest_dir, check_subdir):
    if check_subdir.exists() and len(list(check_subdir.glob('*.npy'))) > 0:
        count = len(list(check_subdir.glob('*.npy')))
        print(f'Already extracted: {check_subdir} ({count} files)')
        return
    print(f'Extracting {zip_path.name}...')
    t0 = time.time()
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(dest_dir)
    count = len(list(check_subdir.glob('*.npy')))
    print(f'Extracted {count} files in {time.time()-t0:.1f}s')

if train_zip.exists():
    extract_if_needed(train_zip, WORK_DIR, TRAIN_NOISY_DIR)
if test_zip.exists():
    extract_if_needed(test_zip, WORK_DIR, TEST_NOISY_DIR)

n_noisy = len(list(TRAIN_NOISY_DIR.glob('*.npy'))) if TRAIN_NOISY_DIR.exists() else 0
n_gt    = len(list(TRAIN_GT_DIR.glob('*.npy')))    if TRAIN_GT_DIR.exists()    else 0
print(f'NoisyLR: {n_noisy} files | GT: {n_gt} files')
assert n_noisy > 0 and n_gt > 0, 'Dataset not found! Check paths.'
assert n_noisy == n_gt, f'Mismatch: {n_noisy} noisy vs {n_gt} GT'

## Step 4: Write Source Files

In [ ]:
%%writefile /content/work_stage1/dataset.py
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import random

class SemconDataset(Dataset):
    def __init__(self, noisy_dir, gt_dir=None, augment=True):
        self.noisy_dir = noisy_dir
        self.gt_dir = gt_dir
        self.augment = augment and (gt_dir is not None)
        self.filenames = sorted([f for f in os.listdir(noisy_dir) if f.endswith('.npy')])
        if len(self.filenames) == 0:
            raise RuntimeError(f'No .npy files found in {noisy_dir}')
        print(f'[Dataset] Found {len(self.filenames)} files | augment={self.augment}')

    def __len__(self): return len(self.filenames)

    def _load_npy(self, path):
        arr = np.load(path).astype(np.float32)
        if arr.max() > 2.0: arr = arr / 255.0
        mn, mx = arr.min(), arr.max()
        if mx > mn: arr = (arr - mn) / (mx - mn)
        else: arr = np.zeros_like(arr)
        return arr

    def _to_tensor(self, arr):
        if arr.ndim == 2: arr = arr[np.newaxis]
        elif arr.ndim == 3: arr = arr.transpose(2, 0, 1)
        return torch.from_numpy(arr.copy())

    def _augment_pair(self, noisy, gt):
        if random.random() > 0.5: noisy, gt = np.fliplr(noisy), np.fliplr(gt)
        if random.random() > 0.5: noisy, gt = np.flipud(noisy), np.flipud(gt)
        k = random.randint(0, 3)
        return np.rot90(noisy, k), np.rot90(gt, k)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        noisy = self._load_npy(os.path.join(self.noisy_dir, fname))
        if self.gt_dir is not None:
            gt = self._load_npy(os.path.join(self.gt_dir, fname))
            if self.augment: noisy, gt = self._augment_pair(noisy, gt)
            return self._to_tensor(noisy), self._to_tensor(gt)
        return self._to_tensor(noisy), fname

class _AugmentedSubset(Dataset):
    def __init__(self, subset, noisy_dir, gt_dir):
        self.subset = subset
        self.aug_dataset = SemconDataset(noisy_dir, gt_dir, augment=True)
    def __len__(self): return len(self.subset)
    def __getitem__(self, idx): return self.aug_dataset[self.subset.indices[idx]]

def get_dataloaders(train_noisy_dir, train_gt_dir, val_split=0.1,
                    batch_size=16, num_workers=4, seed=42):
    pin_mem = torch.cuda.is_available()
    full_ds = SemconDataset(train_noisy_dir, train_gt_dir, augment=False)
    n_val   = max(1, int(len(full_ds) * val_split))
    n_train = len(full_ds) - n_val
    rng = torch.Generator(); rng.manual_seed(seed)
    train_ds, val_ds = torch.utils.data.random_split(full_ds, [n_train, n_val], generator=rng)
    train_aug = _AugmentedSubset(train_ds, train_noisy_dir, train_gt_dir)
    tr = DataLoader(train_aug, batch_size=batch_size, shuffle=True,
                    num_workers=num_workers, pin_memory=pin_mem, drop_last=True)
    vl = DataLoader(val_ds,   batch_size=4, shuffle=False,
                    num_workers=num_workers, pin_memory=pin_mem)
    print(f'[DataLoader] Train: {n_train} | Val: {n_val} | Batch: {batch_size}')
    return tr, vl

In [ ]:
%%writefile /content/work_stage1/model.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(ch, ch, 3, 1, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ch, ch, 3, 1, 1))
        self.act = nn.LeakyReLU(0.2, True)
    def forward(self, x): return self.act(x + self.body(x))

class DepthwiseSepConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, 3, 1, 1, groups=in_ch, bias=False),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.LeakyReLU(0.2, True))
    def forward(self, x): return self.net(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, 1, 1), nn.LeakyReLU(0.2, True), ResBlock(out_ch))
        self.down = nn.Conv2d(out_ch, out_ch, 3, 2, 1)
    def forward(self, x):
        f = self.conv(x); return f, self.down(f)

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.Sequential(nn.Conv2d(in_ch, out_ch*4, 1), nn.PixelShuffle(2))
        self.conv = nn.Sequential(nn.Conv2d(out_ch+skip_ch, out_ch, 3, 1, 1), nn.LeakyReLU(0.2, True), ResBlock(out_ch))
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]: x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([x, skip], 1))

class SemiRestoreNet(nn.Module):
    def __init__(self, base_ch=32):
        super().__init__()
        b = base_ch
        self.enc0       = nn.Sequential(nn.Conv2d(1, b, 3, 1, 1), nn.LeakyReLU(0.2, True))
        self.down1      = DownBlock(b,   b*2)
        self.down2      = DownBlock(b*2, b*4)
        self.down3      = DownBlock(b*4, b*8)
        self.bottleneck = nn.Sequential(DepthwiseSepConv(b*8, b*8), ResBlock(b*8), DepthwiseSepConv(b*8, b*8))
        self.up3        = UpBlock(b*8, b*8, b*4)
        self.up2        = UpBlock(b*4, b*4, b*2)
        self.up1        = UpBlock(b*2, b*2, b)
        self.out_head   = nn.Sequential(
            nn.Conv2d(b, b, 3, 1, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(b, 4, 1), nn.PixelShuffle(2), nn.Sigmoid())
    def forward(self, x):
        e0 = self.enc0(x)
        s1, d1 = self.down1(e0)
        s2, d2 = self.down2(d1)
        s3, d3 = self.down3(d2)
        b = self.bottleneck(d3)
        u3 = self.up3(b, s3); u2 = self.up2(u3, s2); u1 = self.up1(u2, s1)
        return self.out_head(u1)

def build_model(base_ch=32):
    m = SemiRestoreNet(base_ch)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'[Model] SemiRestoreNet | base_ch={base_ch} | params={n:,}')
    return m

In [ ]:
%%writefile /content/work_stage1/losses.py
import torch, torch.nn as nn, torch.nn.functional as F

def _gauss_kernel(size=11, sigma=1.5):
    c = torch.arange(size, dtype=torch.float32) - size//2
    g = torch.exp(-(c**2)/(2*sigma**2))
    k = g.outer(g); return (k/k.sum()).unsqueeze(0).unsqueeze(0)

class SSIMLoss(nn.Module):
    def __init__(self, ws=11):
        super().__init__()
        self.register_buffer('k', _gauss_kernel(ws))
        self.ws, self.C1, self.C2 = ws, 1e-4, 9e-4
    def forward(self, p, t):
        pad, k = self.ws//2, self.k.to(p.device)
        mx = F.conv2d(p, k, padding=pad); my = F.conv2d(t, k, padding=pad)
        mx2,my2,mxy = mx*mx, my*my, mx*my
        sx = F.conv2d(p*p, k, padding=pad)-mx2; sy = F.conv2d(t*t, k, padding=pad)-my2
        sxy = F.conv2d(p*t, k, padding=pad)-mxy
        return 1 - (((2*mxy+self.C1)*(2*sxy+self.C2))/((mx2+my2+self.C1)*(sx+sy+self.C2))).mean()

class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__(); self.vgg = None
        try:
            import torchvision.models as M
            v = M.vgg16(weights=M.VGG16_Weights.IMAGENET1K_V1)
            self.vgg = nn.Sequential(*list(v.features.children())[:10]).eval()
            for p in self.vgg.parameters(): p.requires_grad_(False)
            print('[Loss] VGG16 perceptual OK')
        except Exception as e: print(f'[Loss] VGG unavailable: {e}')
    def forward(self, p, t):
        if self.vgg is None: return torch.tensor(0., device=p.device)
        self.vgg = self.vgg.to(p.device)
        return F.l1_loss(self.vgg(p.repeat(1,3,1,1)), self.vgg(t.repeat(1,3,1,1)))

class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.6, beta=0.3, gamma=0.1):
        super().__init__()
        self.a, self.b, self.g = alpha, beta, gamma
        self.l1 = nn.L1Loss(); self.ssim = SSIMLoss(); self.perc = PerceptualLoss()
        print(f'[Loss] {alpha}*L1 + {beta}*SSIM + {gamma}*Perceptual')
    def forward(self, p, t):
        l1 = self.l1(p,t); ss = self.ssim(p,t); pc = self.perc(p,t)
        tot = self.a*l1 + self.b*ss + self.g*pc
        return tot, {'l1':l1.item(),'ssim':ss.item(),'perceptual':pc.item(),'total':tot.item()}

In [ ]:
%%writefile /content/work_stage1/metrics.py
import torch, torch.nn.functional as F

def compute_psnr(pred, target, max_val=1.0):
    with torch.no_grad():
        mse = F.mse_loss(pred, target, reduction='none').view(pred.size(0),-1).mean(1)
        return (10*torch.log10(max_val**2/(mse+1e-8))).mean().item()

def compute_ssim(pred, target, ws=11, sigma=1.5):
    C1,C2 = 1e-4,9e-4; pad=ws//2
    c = torch.arange(ws,dtype=torch.float32,device=pred.device)-ws//2
    g = torch.exp(-(c**2)/(2*sigma**2)); k=g.outer(g); k=(k/k.sum()).unsqueeze(0).unsqueeze(0)
    with torch.no_grad():
        mx=F.conv2d(pred,k,padding=pad); my=F.conv2d(target,k,padding=pad)
        mx2,my2,mxy=mx*mx,my*my,mx*my
        sx=F.conv2d(pred*pred,k,padding=pad)-mx2; sy=F.conv2d(target*target,k,padding=pad)-my2
        sxy=F.conv2d(pred*target,k,padding=pad)-mxy
        return (((2*mxy+C1)*(2*sxy+C2))/((mx2+my2+C1)*(sx+sy+C2)+1e-8)).mean().item()

## Step 5: Configure & Train

In [ ]:
CONFIG = {
    'train_noisy_dir': str(TRAIN_NOISY_DIR),
    'train_gt_dir'   : str(TRAIN_GT_DIR),
    'output_dir'     : str(OUTPUT_DIR),
    'epochs'         : 100,
    'batch_size'     : 64,     # Increased to 64 for faster GPU saturation
    'lr'             : 6e-4,   # Scaled up because of larger batch size
    'lr_min'         : 1e-6,
    'warmup_epochs'  : 5,
    'val_split'      : 0.10,
    'base_ch'        : 32,
    'loss_alpha'     : 0.60,
    'loss_beta'      : 0.30,
    'loss_gamma'     : 0.10,
    'num_workers'    : 2,      # Colab has 2 vCPUs, 4 creates bottleneck
    'amp'            : True,
    'seed'           : 42,
}
print('Config ready:', CONFIG)

In [ ]:
import sys, random, math, json, shutil, time
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.amp import autocast

sys.path.insert(0, '/content/work_stage1')
from dataset import get_dataloaders
from model   import build_model
from losses  import CombinedLoss
from metrics import compute_psnr, compute_ssim

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def get_lr_schedule(opt, epochs, warmup, lr_min, lr_max):
    def fn(ep):
        if ep < warmup: return (ep+1)/max(warmup,1)
        p = (ep-warmup)/max(epochs-warmup,1)
        return lr_min/lr_max + (1-lr_min/lr_max)*0.5*(1+math.cos(math.pi*p))
    return optim.lr_scheduler.LambdaLR(opt, fn)

class Meter:
    def __init__(self): self.sum=self.count=0
    def update(self,v,n=1): self.sum+=v*n; self.count+=n
    @property
    def avg(self): return self.sum/max(self.count,1)

def save_ckpt(m, opt, sch, ep, best, path):
    m_state = m._orig_mod.state_dict() if hasattr(m, '_orig_mod') else m.state_dict()
    torch.save({'epoch':ep,'model_state_dict':m_state,
                'optimizer_state_dict':opt.state_dict(),
                'scheduler_state_dict':sch.state_dict(),'best_psnr':best}, path)

def load_ckpt(m, opt, sch, path, dev):
    c = torch.load(path, map_location=dev)
    if hasattr(m, '_orig_mod'): m._orig_mod.load_state_dict(c['model_state_dict'])
    else: m.load_state_dict(c['model_state_dict'])
    opt.load_state_dict(c['optimizer_state_dict'])
    sch.load_state_dict(c['scheduler_state_dict'])
    return c['epoch'], c['best_psnr']

def train_epoch(model, loader, opt, loss_fn, scaler, dev, amp):
    model.train()
    meters = {k: Meter() for k in ['total','l1','ssim','perceptual']}
    t0 = time.time()
    for i,(noisy,gt) in enumerate(loader):
        noisy,gt = noisy.to(dev,non_blocking=True), gt.to(dev,non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast(device_type='cuda' if amp else 'cpu', enabled=amp):
            pred = model(noisy)
        # Compute loss in float32 to prevent AMP NaN instability
        loss,comp = loss_fn(pred.float(), gt.float())
        if amp:
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt); scaler.update()
        else:
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        n = noisy.size(0)
        for k,v in comp.items(): meters[k].update(v,n)
        if (i+1)%20==0:
            print(f'  [{i+1:3d}/{len(loader)}] loss={meters["total"].avg:.4f} '
                  f'l1={meters["l1"].avg:.4f} ssim={meters["ssim"].avg:.4f} t={time.time()-t0:.0f}s')
    return {k:m.avg for k,m in meters.items()}

@torch.no_grad()
def validate(model, loader, dev):
    model.eval()
    pm,sm = Meter(),Meter()
    for noisy,gt in loader:
        noisy,gt = noisy.to(dev),gt.to(dev)
        pred = model(noisy).clamp(0,1)
        pm.update(compute_psnr(pred,gt),noisy.size(0))
        sm.update(compute_ssim(pred,gt),noisy.size(0))
    return pm.avg, sm.avg

def train(cfg):
    seed_everything(cfg['seed'])
    torch.backends.cudnn.benchmark = True  # Auto-tunes convolutions for ~15% speedup
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    amp = cfg['amp'] and dev.type=='cuda'
    print(f'[Train] Device: {dev} | AMP: {amp}')

    tr,vl = get_dataloaders(cfg['train_noisy_dir'],cfg['train_gt_dir'],
                            cfg['val_split'],cfg['batch_size'],cfg['num_workers'],cfg['seed'])
    model   = build_model(cfg['base_ch']).to(dev)
    if hasattr(torch, 'compile'):
        print('[Train] Compiling model with torch.compile() for speed...')
        model = torch.compile(model)
    opt     = optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=1e-4)
    sch     = get_lr_schedule(opt, cfg['epochs'], cfg['warmup_epochs'], cfg['lr_min'], cfg['lr'])
    scaler  = torch.amp.GradScaler('cuda', enabled=amp)
    loss_fn = CombinedLoss(cfg['loss_alpha'],cfg['loss_beta'],cfg['loss_gamma']).to(dev)

    out      = Path(cfg['output_dir']); out.mkdir(parents=True, exist_ok=True)
    ckpt_best = out/'best_model.pth'; ckpt_drv = out/'last_model.pth'
    ckpt_loc  = Path('/content/last_model.pth')
    log_path  = out/'train_log.json'

    start_ep, best_psnr = 0, 0.0
    if ckpt_drv.exists():
        print(f'Resuming from {ckpt_drv}')
        shutil.copy(ckpt_drv, ckpt_loc)
        start_ep, best_psnr = load_ckpt(model,opt,sch,ckpt_loc,dev)
        start_ep += 1

    log=[]; t_total=time.time()
    print(f'{"="*60}
  Training | {cfg["epochs"]} epochs | batch={cfg["batch_size"]}
{"="*60}')

    for ep in range(start_ep, cfg['epochs']):
        lr_now = opt.param_groups[0]['lr']
        t_ep   = time.time()
        tm     = train_epoch(model,tr,opt,loss_fn,scaler,dev,amp)
        vp,vs  = validate(model,vl,dev)
        sch.step()
        ep_t   = time.time()-t_ep
        eta    = (cfg['epochs']-ep-1)*ep_t/60
        log.append({'epoch':ep,'lr':lr_now,'train_loss':tm['total'],'val_psnr':vp,'val_ssim':vs,'time':ep_t})
        print(f'Epoch [{ep+1:03d}/{cfg["epochs"]}] loss={tm["total"]:.4f} PSNR={vp:.2f}dB SSIM={vs:.4f} lr={lr_now:.1e} t={ep_t:.0f}s ETA={eta:.0f}min')

        save_ckpt(model,opt,sch,ep,best_psnr,ckpt_loc)
        if (ep+1)%5==0 or ep==cfg['epochs']-1:
            shutil.copy(ckpt_loc, ckpt_drv); print('  Checkpoint saved to Drive')
        if vp > best_psnr:
            best_psnr=vp; save_ckpt(model,opt,sch,ep,best_psnr,ckpt_best)
            print(f'  NEW BEST PSNR: {best_psnr:.2f} dB')
        with open(log_path,'w') as f: json.dump(log,f,indent=2)

    print(f'{"="*60}
  Training done in {(time.time()-t_total)/60:.1f} min | Best PSNR: {best_psnr:.2f} dB
{"="*60}')
    return log

train_log = train(CONFIG)

## Step 6: Plot Training Curves

In [ ]:
if len(train_log) == 0:
    print("WARNING: train_log is empty! (Training was skipped because it reached max epochs). Clear your checkpoints to train from scratch.")
else:
    import matplotlib.pyplot as plt

    eps   = [e['epoch']+1    for e in train_log]
    loss  = [e['train_loss'] for e in train_log]
    psnrs = [e['val_psnr']   for e in train_log]
    ssims = [e['val_ssim']   for e in train_log]

    fig, ax = plt.subplots(1, 3, figsize=(15,4))
    fig.suptitle('SemiRestoreNet — Training Curves', fontsize=13, fontweight='bold')
    ax[0].plot(eps, loss, 'b-'); ax[0].set_title('Train Loss'); ax[0].set_xlabel('Epoch'); ax[0].grid(alpha=0.3)
    ax[1].plot(eps, psnrs, 'g-'); ax[1].axhline(max(psnrs), color='g', ls='--', alpha=0.5, label=f'Best {max(psnrs):.2f}dB')
    ax[1].set_title('Val PSNR (dB)'); ax[1].set_xlabel('Epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
    ax[2].plot(eps, ssims, 'r-'); ax[2].axhline(max(ssims), color='r', ls='--', alpha=0.5, label=f'Best {max(ssims):.4f}')
    ax[2].set_title('Val SSIM'); ax[2].set_xlabel('Epoch'); ax[2].legend(); ax[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Best PSNR: {max(psnrs):.2f} dB | Best SSIM: {max(ssims):.4f}')

## Step 7: Visualize Sample Predictions

In [ ]:
import matplotlib.pyplot as plt
import torch, numpy as np
from model   import build_model
from dataset import SemconDataset
from metrics import compute_psnr, compute_ssim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = build_model(CONFIG['base_ch']).to(device)
ckpt   = torch.load(OUTPUT_DIR/'best_model.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict']); model.eval()
print(f'Best model: epoch {ckpt["epoch"]+1} | PSNR={ckpt["best_psnr"]:.2f} dB')

ds = SemconDataset(CONFIG['train_noisy_dir'], CONFIG['train_gt_dir'], augment=False)
idxs = [0, 50, 100, 200]
fig, axes = plt.subplots(len(idxs), 3, figsize=(12, 4*len(idxs)))
fig.suptitle('Noisy LR Input | Prediction | Ground Truth', fontsize=12, fontweight='bold')

for row, idx in enumerate(idxs):
    nt, gt = ds[idx]
    with torch.no_grad(): pt = model(nt.unsqueeze(0).to(device)).squeeze(0).cpu().clamp(0,1)
    pv = compute_psnr(pt.unsqueeze(0), gt.unsqueeze(0))
    sv = compute_ssim(pt.unsqueeze(0), gt.unsqueeze(0))
    axes[row,0].imshow(nt.squeeze().numpy(), cmap='gray', vmin=0, vmax=1); axes[row,0].set_title(f'Noisy LR #{idx}'); axes[row,0].axis('off')
    axes[row,1].imshow(pt.squeeze().numpy(), cmap='gray', vmin=0, vmax=1); axes[row,1].set_title(f'Pred | PSNR={pv:.2f}dB SSIM={sv:.4f}'); axes[row,1].axis('off')
    axes[row,2].imshow(gt.squeeze().numpy(), cmap='gray', vmin=0, vmax=1); axes[row,2].set_title('GT'); axes[row,2].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR/'predictions_viz.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Export ONNX & Download

In [ ]:
import torch, onnxruntime as ort
from model import build_model

cpu_model = build_model(CONFIG['base_ch'])
ckpt = torch.load(OUTPUT_DIR/'best_model.pth', map_location='cpu')
cpu_model.load_state_dict(ckpt['model_state_dict']); cpu_model.eval()

dummy = torch.randn(1,1,128,128)
onnx_path = str(OUTPUT_DIR/'semi_restore_net.onnx')
torch.onnx.export(cpu_model, dummy, onnx_path,
    input_names=['noisy_lr'], output_names=['clean_hr'],
    dynamic_axes={'noisy_lr':{0:'batch'},'clean_hr':{0:'batch'}},
    opset_version=17)

sess = ort.InferenceSession(onnx_path)
out  = sess.run(None, {'noisy_lr': dummy.numpy()})
print(f'ONNX OK | output shape: {out[0].shape} | size: {Path(onnx_path).stat().st_size/1e6:.1f} MB')

from google.colab import files
files.download(str(OUTPUT_DIR/'best_model.pth'))

## Step 9: Generate Test Predictions (for submission)

In [ ]:
import zipfile, torch, numpy as np
from tqdm.notebook import tqdm
from model import build_model

if not TEST_NOISY_DIR.exists():
    print('Test NoisyLR not found. Extract Test_NoisyLR.zip first.')
else:
    PRED_DIR = OUTPUT_DIR/'predictions'; PRED_DIR.mkdir(exist_ok=True)
    device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model    = build_model(CONFIG['base_ch']).to(device)
    ckpt     = torch.load(OUTPUT_DIR/'best_model.pth', map_location=device)
    model.load_state_dict(ckpt['model_state_dict']); model.eval()

    test_files = sorted(TEST_NOISY_DIR.glob('*.npy'))
    print(f'Running inference on {len(test_files)} test images...')

    for fp in tqdm(test_files):
        arr = np.load(fp).astype(np.float32)
        if arr.max()>2.: arr/=255.
        mn,mx = arr.min(),arr.max()
        if mx>mn: arr=(arr-mn)/(mx-mn)
        if arr.ndim==2: arr=arr[np.newaxis]
        t = torch.from_numpy(arr).unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(t).squeeze(0).cpu().clamp(0,1).numpy().squeeze()
        np.save(PRED_DIR/fp.name, pred.astype(np.float32))

    zip_path = OUTPUT_DIR/'predictions.zip'
    with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(PRED_DIR.glob('*.npy')): zf.write(f, f.name)
    print(f'Done! {len(test_files)} predictions zipped ({zip_path.stat().st_size/1e6:.1f} MB)')

    from google.colab import files
    files.download(str(zip_path))